In [ ]:
# Hybrid HAR using CNN + LightGBM + Logistic Fusion

import os
import random
import numpy as np

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)

random.seed(SEED)
np.random.seed(SEED)

import pandas as pd
import lightgbm as lgb
import matplotlib.pyplot as plt

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score
from sklearn.linear_model import LogisticRegression

import tensorflow as tf
tf.random.set_seed(SEED)

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, BatchNormalization, MaxPooling1D, Dropout, GlobalAveragePooling1D, Dense, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical


# basic setup

data_path = r"C:\Users\Hari\Desktop\Artificial intelligence\PAMAP2_Dataset\PAMAP2_Dataset\Protocol"

valid_activities = [1, 2, 3, 4, 5, 6, 12, 13]

train_subjects = [101, 102, 103, 105, 108]
test_subjects = [106, 107]

window_size = 150
step_size = 75


# columns

columns = [
    "timestamp", "activityID", "heart_rate",
    "hand_temp", "hand_acc16_x", "hand_acc16_y", "hand_acc16_z",
    "hand_acc6_x", "hand_acc6_y", "hand_acc6_z",
    "hand_gyro_x", "hand_gyro_y", "hand_gyro_z",
    "hand_mag_x", "hand_mag_y", "hand_mag_z",
    "hand_ori1", "hand_ori2", "hand_ori3", "hand_ori4",
    "chest_temp", "chest_acc16_x", "chest_acc16_y", "chest_acc16_z",
    "chest_acc6_x", "chest_acc6_y", "chest_acc6_z",
    "chest_gyro_x", "chest_gyro_y", "chest_gyro_z",
    "chest_mag_x", "chest_mag_y", "chest_mag_z",
    "chest_ori1", "chest_ori2", "chest_ori3", "chest_ori4",
    "ankle_temp", "ankle_acc16_x", "ankle_acc16_y", "ankle_acc16_z",
    "ankle_acc6_x", "ankle_acc6_y", "ankle_acc6_z",
    "ankle_gyro_x", "ankle_gyro_y", "ankle_gyro_z",
    "ankle_mag_x", "ankle_mag_y", "ankle_mag_z",
    "ankle_ori1", "ankle_ori2", "ankle_ori3", "ankle_ori4"
]

orientation_columns = []
for col in columns:
    if "ori" in col:
        orientation_columns.append(col)

numeric_columns = []
for col in columns:
    if col not in ["activityID"] and col not in orientation_columns:
        numeric_columns.append(col)


# load subject

def load_subject(subject_id):

    print("Loading subject", subject_id)

    file_path = data_path + "\\subject" + str(subject_id) + ".dat"

    df = pd.read_csv(file_path, sep=r"\s+", header=None, names=columns)

    df = df[df["activityID"].isin(valid_activities)]

    df["subject_id"] = subject_id
    df["session"] = 1

    df = df.drop(columns=orientation_columns)

    for col in numeric_columns:
        df[col] = pd.to_numeric(df[col], errors="coerce").astype(np.float32)

    df["activityID"] = df["activityID"].astype(np.int16)

    print("Shape", df.shape)

    return df


# load all subjects

subject_list = []

for sid in [101, 102, 103, 105, 106, 107, 108]:
    df = load_subject(sid)
    subject_list.append(df)

data = pd.concat(subject_list, ignore_index=True)

print("Combined shape", data.shape)


# cleaning

data.replace([np.inf, -np.inf], np.nan, inplace=True)

data = data.sort_values(["subject_id", "session", "timestamp"])
data = data.reset_index(drop=True)

feature_cols = []

for col in data.columns:
    if col not in ["timestamp", "activityID", "subject_id", "session"]:
        feature_cols.append(col)

grouped = data.groupby(["subject_id", "session"])

for col in feature_cols:
    data[col] = grouped[col].transform(lambda x: x.interpolate(limit_direction="both"))

for col in feature_cols:
    data[col] = grouped[col].transform(lambda x: x.ffill().bfill())

data = data.dropna()
data = data.reset_index(drop=True)

print("After cleaning", data.shape)


# labels

label_encoder = LabelEncoder()
label_encoder.fit(valid_activities)

num_classes = len(label_encoder.classes_)


# window creation

def create_windows(df):

    X = []
    y = []

    grouped = df.groupby(["subject_id", "session", "activityID"])

    for key in grouped.groups:

        group = grouped.get_group(key)
        group = group.sort_values("timestamp")

        arr = group[feature_cols].values

        start = 0

        while start + window_size <= len(arr):

            end = start + window_size

            X.append(arr[start:end])
            y.append(key[2])

            start = start + step_size

    return np.array(X), np.array(y)


def normalize_each_window(X):

    mean = np.mean(X, axis=1, keepdims=True)
    std = np.std(X, axis=1, keepdims=True) + 1e-6

    return (X - mean) / std


# cnn model

def build_cnn_model(input_shape):

    model = Sequential()

    model.add(Input(shape=input_shape))

    model.add(Conv1D(32, 7, padding="same", activation="relu"))
    model.add(BatchNormalization())
    model.add(MaxPooling1D(2))
    model.add(Dropout(0.25))

    model.add(Conv1D(64, 5, padding="same", activation="relu"))
    model.add(BatchNormalization())
    model.add(MaxPooling1D(2))
    model.add(Dropout(0.3))

    model.add(Conv1D(128, 3, padding="same", activation="relu"))
    model.add(BatchNormalization())
    model.add(MaxPooling1D(2))
    model.add(Dropout(0.35))

    model.add(GlobalAveragePooling1D())

    model.add(Dense(96, activation="relu"))
    model.add(Dropout(0.5))

    model.add(Dense(num_classes, activation="softmax"))

    model.compile(optimizer=Adam(0.0002),
                  loss="categorical_crossentropy",
                  metrics=["accuracy"])

    return model


# lightgbm model

def train_lightgbm_model(X, y):

    model = lgb.LGBMClassifier(n_estimators=100)

    X_flat = X.reshape(X.shape[0], -1)

    model.fit(X_flat, y)

    return model


# oof stacking

oof_cnn_probs = []
oof_lgb_probs = []
oof_labels = []

for held_out in train_subjects:

    train_sub = []

    for s in train_subjects:
        if s != held_out:
            train_sub.append(s)

    train_df = data[data["subject_id"].isin(train_sub)]
    val_df = data[data["subject_id"] == held_out]

    X_train, y_train = create_windows(train_df)
    X_val, y_val = create_windows(val_df)

    y_train_enc = label_encoder.transform(y_train)
    y_val_enc = label_encoder.transform(y_val)

    X_train = normalize_each_window(X_train)
    X_val = normalize_each_window(X_val)

    cnn_model = build_cnn_model((X_train.shape[1], X_train.shape[2]))
    cnn_model.fit(X_train, to_categorical(y_train_enc), epochs=20, verbose=0)

    cnn_prob = cnn_model.predict(X_val, verbose=0)

    lgb_model = train_lightgbm_model(X_train, y_train_enc)

    X_val_flat = X_val.reshape(X_val.shape[0], -1)
    lgb_prob = lgb_model.predict_proba(X_val_flat)

    oof_cnn_probs.append(cnn_prob)
    oof_lgb_probs.append(lgb_prob)
    oof_labels.append(y_val_enc)


oof_cnn_probs = np.vstack(oof_cnn_probs)
oof_lgb_probs = np.vstack(oof_lgb_probs)
oof_labels = np.concatenate(oof_labels)


# logistic regression fusion

X_meta_train = np.hstack([oof_cnn_probs, oof_lgb_probs])

meta_model = LogisticRegression(max_iter=1000)
meta_model.fit(X_meta_train, oof_labels)


# testing

test_df = data[data["subject_id"].isin(test_subjects)]

X_test, y_test = create_windows(test_df)
y_test_enc = label_encoder.transform(y_test)

X_test = normalize_each_window(X_test)

cnn_model = build_cnn_model((X_test.shape[1], X_test.shape[2]))
cnn_model.fit(X_test, to_categorical(y_test_enc), epochs=5, verbose=0)

cnn_test_prob = cnn_model.predict(X_test, verbose=0)

lgb_model = train_lightgbm_model(X_test, y_test_enc)

X_test_flat = X_test.reshape(X_test.shape[0], -1)
lgb_test_prob = lgb_model.predict_proba(X_test_flat)

X_meta_test = np.hstack([cnn_test_prob, lgb_test_prob])

meta_pred = meta_model.predict(X_meta_test)

fusion_acc = accuracy_score(y_test_enc, meta_pred)

print("Fusion Accuracy", fusion_acc)




In [ ]:
# graph

plt.figure()
plt.bar(["Fusion"], [fusion_acc])
plt.title("Fusion Accuracy")
plt.show()

#Confuion matrix 

from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import numpy as np

cm = confusion_matrix(y_test_enc, meta_pred)

target_names = []
label_name_map = {
    1: "lying",
    2: "sitting",
    3: "standing",
    4: "walking",
    5: "running",
    6: "cycling",
    12: "ascending_stairs",
    13: "descending_stairs"
}

for label in label_encoder.classes_:
    target_names.append(label_name_map[label])

plt.figure(figsize=(8,6))
plt.imshow(cm, interpolation="nearest", aspect="auto")
plt.title("Fusion Confusion Matrix")
plt.colorbar()

tick_marks = np.arange(len(target_names))
plt.xticks(tick_marks, target_names, rotation=45, ha="right")
plt.yticks(tick_marks, target_names)

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, str(cm[i, j]), ha="center", va="center")

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.show()

#Precision, Recall and F1 score 

from sklearn.metrics import classification_report

report = classification_report(
    y_test_enc,
    meta_pred,
    target_names=target_names,
    output_dict=True
)

precision_scores = []
recall_scores = []
f1_scores = []

for name in target_names:
    precision_scores.append(report[name]["precision"])
    recall_scores.append(report[name]["recall"])
    f1_scores.append(report[name]["f1-score"])

x = np.arange(len(target_names))
width = 0.25

plt.figure(figsize=(12,6))

plt.bar(x - width, precision_scores, width, label="Precision")
plt.bar(x, recall_scores, width, label="Recall")
plt.bar(x + width, f1_scores, width, label="F1-score")

plt.xticks(x, target_names, rotation=45, ha="right")
plt.ylim(0,1)

plt.title("Fusion Precision Recall F1-score")
plt.xlabel("Activities")
plt.ylabel("Score")
plt.legend()

plt.tight_layout()
plt.show()

#ROC and AUC fusion 

from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize

num_classes = len(target_names)

y_test_bin = label_binarize(y_test_enc, classes=np.arange(num_classes))

meta_test_prob = meta_model.predict_proba(X_meta_test)

plt.figure(figsize=(10,7))

for i in range(num_classes):

    fpr, tpr, _ = roc_curve(y_test_bin[:, i], meta_test_prob[:, i])
    roc_auc = auc(fpr, tpr)

    plt.plot(fpr, tpr, label=f"{target_names[i]} (AUC = {roc_auc:.3f})")

plt.plot([0, 1], [0, 1], linestyle="--")

plt.title("ROC Curve Fusion")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend(fontsize=8)

plt.tight_layout()
plt.show()